# Pancytopenia Detection — ClinicalBERT + XGBoost Pipeline

This notebook implements the full two-stage multimodal ML pipeline:

**Stage 1 — Fine-tune ClinicalBERT on clinical notes**
- Model: `emilyalsentzer/Bio_ClinicalBERT` (TensorFlow/Keras)
- Freeze bottom 10 of 12 transformer layers
- Fine-tune top 2 layers + classification head
- 5-fold OOF predictions for training patients (same splits as XGBoost)
- Output: `bert_note_prob` — one probability per patient from notes alone

**Stage 2 — XGBoost with enriched features**
- Add `bert_note_prob` as feature 140 alongside 139 tabular features
- Optuna hyperparameter tuning, 3-fold CV on train only
- OOF MCC-optimized threshold — test set never used for threshold selection
- SHAP analysis — does `bert_note_prob` contribute?
- Final evaluation on test set used exactly once

**Why this architecture:**
Feeding 768 BERT embedding dimensions directly into XGBoost is wrong — XGBoost splits on individual dimensions and destroys the geometric structure of embeddings. Instead, fine-tuned BERT produces one meaningful probability per patient. XGBoost can split on this correctly: `bert_note_prob > 0.6`. SHAP works cleanly. EPV unchanged (139 -> 140 features).

**OOF requirement:** Both stages use the same 5-fold stratified splits (SEED=42). Each training patient gets a BERT prediction from a model that never saw them. This prevents the XGBoost from learning an inflated BERT signal.

## 1. Setup and Imports

In [ ]:
# Install if needed
# !pip install tensorflow transformers datasets --quiet

import os, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    confusion_matrix, matthews_corrcoef,
    precision_recall_curve, roc_curve
)
import xgboost as xgb
import optuna
import shap
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED      = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus}')
    print('ClinicalBERT fine-tuning will use GPU.')
else:
    print('WARNING: No GPU detected. Running on CPU.')
    print('Fine-tuning will be slow (~30-60 min per fold on CPU).')
    print('Proceeding as requested.')
print(f'TensorFlow version: {tf.__version__}')

## 2. Load Data

Load the tabular feature matrix and the de-identified clinical notes.

**Notes file:** output of `phi_screening_pipeline.ipynb` — use `note_text_clean` column.

**Tabular data:** pre-built feature matrix with 139 features, train/test split already applied.

In [ ]:
# ── Load tabular feature matrix ──────────────────────────────────────────
TABULAR_PATH = 'xgb_clean.pkl'
NOTES_PATH   = 'notes_clean.csv'

with open(TABULAR_PATH, 'rb') as f:
    d = pickle.load(f)

Xtr_tab = d['Xtr']   # train tabular features (7953, 139)
Xte_tab = d['Xte']   # test tabular features  (1989, 139)
ytr     = d['ytr']   # train labels
yte     = d['yte']   # test labels
fc      = d['fc']    # feature column names
spw     = d['spw']   # scale_pos_weight for XGBoost

print(f'Tabular train: {Xtr_tab.shape} | positives: {ytr.sum()}')
print(f'Tabular test:  {Xte_tab.shape} | positives: {yte.sum()}')
print(f'scale_pos_weight: {spw:.1f}')

# ── Load notes ────────────────────────────────────────────────────────────
notes_df = pd.read_csv(NOTES_PATH)
print(f'Notes loaded: {len(notes_df):,} rows')
print(f'Columns: {notes_df.columns.tolist()}')

# Use note_text_clean (de-identified) if available, else note_text
text_col = 'note_text_clean' if 'note_text_clean' in notes_df.columns else 'note_text'
print(f'Using text column: {text_col}')

# ── Load cohort to get all visit IDs with labels ──────────────────────────
import pandas as pd
basic  = pd.read_csv('BasicData_Technical_Interview_FINAL.csv', index_col=0)
obs_df = pd.read_csv('Observations_Technical_Interview_FINAL__1_.csv', index_col=0)
basic['label'] = basic['result'].notna().astype(int)
cohort = basic[basic['visit_id'].isin(set(obs_df['visit_id'].unique()))].copy().reset_index(drop=True)

# Merge notes with cohort labels
notes_labeled = cohort[['visit_id','label']].merge(notes_df[['visit_id', text_col]],
                                                    on='visit_id', how='left')
notes_labeled['note_text'] = notes_labeled[text_col].fillna('')

print(f'Notes matched to cohort: {notes_labeled.visit_id.nunique():,}')
print(f'Notes missing:           {notes_labeled.note_text.eq("").sum()}')

## 3. Define Train/Test Split

**Critical:** Both BERT and XGBoost must use the exact same fold splits.
Same SEED=42, same stratification. This ensures no patient in any fold
gets a BERT prediction from a model trained on XGBoost's validation data.

In [ ]:
# Recreate the same train/test split used for tabular features
train_cohort, test_cohort = train_test_split(
    cohort, test_size=0.2, random_state=SEED, stratify=cohort['label']
)
train_visits = set(train_cohort['visit_id'])
test_visits  = set(test_cohort['visit_id'])

# Get note text and labels aligned to train/test split
train_notes = notes_labeled[notes_labeled['visit_id'].isin(train_visits)].reset_index(drop=True)
test_notes  = notes_labeled[notes_labeled['visit_id'].isin(test_visits)].reset_index(drop=True)

# Align row order to match tabular feature matrix row order
# We need bert_note_prob[i] to correspond to Xtr_tab.iloc[i]
v2i_tr = {vid: i for i, vid in enumerate(train_notes['visit_id'])}
v2i_te = {vid: i for i, vid in enumerate(test_notes['visit_id'])}

train_texts  = train_notes['note_text'].tolist()
test_texts   = test_notes['note_text'].tolist()
ytr_bert     = train_notes['label'].values
yte_bert     = test_notes['label'].values

print(f'Train notes: {len(train_texts)} | positives: {ytr_bert.sum()}')
print(f'Test notes:  {len(test_texts)}  | positives: {yte_bert.sum()}')
print()
print('Verifying label alignment between tabular and notes...')
assert len(train_texts) == len(Xtr_tab), 'Train size mismatch'
assert len(test_texts)  == len(Xte_tab), 'Test size mismatch'
print('Alignment verified.')

## 4. Tokenizer and Dataset Preparation

Load the ClinicalBERT tokenizer. Tokenize all notes with max 512 tokens.

**512 token limit:** ClinicalBERT's maximum input length. Our notes average ~650 tokens. We take the first 512 — this captures the chief complaint, HPI, PMH, and medications, which is the most predictive content for our task.

**Truncation:** `truncation=True, max_length=512` handles this automatically.

In [ ]:
BERT_MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LEN         = 512
BATCH_SIZE      = 16

print(f'Loading tokenizer: {BERT_MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
print('Tokenizer loaded.')


def tokenize_texts(texts, tokenizer, max_len=512):
    """
    Tokenize a list of texts for ClinicalBERT input.
    Returns a dict with input_ids and attention_mask as numpy arrays.
    """
    encoded = tokenizer(
        texts,
        max_length      = max_len,
        truncation      = True,         # truncate to first 512 tokens
        padding         = 'max_length', # pad shorter notes to 512
        return_tensors  = 'np'          # return numpy arrays for TF dataset
    )
    return encoded['input_ids'], encoded['attention_mask']


def make_tf_dataset(input_ids, attention_mask, labels=None,
                    batch_size=16, shuffle=False):
    """
    Create a TensorFlow Dataset from tokenized inputs.
    Labels are optional (omit for prediction-only).
    """
    if labels is not None:
        dataset = tf.data.Dataset.from_tensor_slices((
            {'input_ids': input_ids, 'attention_mask': attention_mask},
            labels
        ))
    else:
        dataset = tf.data.Dataset.from_tensor_slices(
            {'input_ids': input_ids, 'attention_mask': attention_mask}
        )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=SEED)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# Tokenize all texts upfront to avoid repeating in each fold
print('Tokenizing training notes...')
train_ids, train_masks = tokenize_texts(train_texts, tokenizer)
print(f'Train tokenized: {train_ids.shape}')

print('Tokenizing test notes...')
test_ids, test_masks   = tokenize_texts(test_texts, tokenizer)
print(f'Test tokenized:  {test_ids.shape}')
print()
print(f'Token coverage check (% notes exceeding 512 tokens):')
lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
pct_truncated = sum(l > 512 for l in lengths) / len(lengths) * 100
print(f'  ~{pct_truncated:.1f}% of notes truncated (sample of 200)')

## 5. ClinicalBERT Model Architecture

We use `TFAutoModelForSequenceClassification` — a ClinicalBERT model with a binary classification head attached on top of the [CLS] token embedding.

**Freezing strategy:**
- Freeze bottom 10 of 12 transformer layers
- Fine-tune top 2 layers + classification head
- Preserves pre-trained clinical language knowledge in lower layers
- Adapts upper layers to pancytopenia prediction task
- Prevents catastrophic forgetting on our small dataset (142 positives)

**Class imbalance:** Handled via `class_weight` in `model.fit()` — equivalent to `scale_pos_weight` in XGBoost but implemented as weighted loss.

In [ ]:
def build_bert_model(bert_model_name, freeze_layers=10, learning_rate=2e-5):
    """
    Build fine-tunable ClinicalBERT binary classifier.

    freeze_layers: number of transformer layers to freeze (bottom N of 12)
    learning_rate: small LR for BERT layers to prevent catastrophic forgetting
    """
    # Load pre-trained ClinicalBERT with binary classification head
    bert = TFAutoModelForSequenceClassification.from_pretrained(
        bert_model_name,
        num_labels = 1   # single output — binary classification
    )

    # Freeze bottom N transformer layers
    # Layer structure: embeddings + 12 encoder layers + pooler
    bert.bert.embeddings.trainable = False
    for i, layer in enumerate(bert.bert.encoder.layer):
        if i < freeze_layers:
            layer.trainable = False
        # Top (12 - freeze_layers) layers remain trainable

    # Count trainable vs frozen parameters
    trainable     = sum(tf.size(v).numpy() for v in bert.trainable_variables)
    non_trainable = sum(tf.size(v).numpy() for v in bert.non_trainable_variables)
    print(f'Trainable parameters:   {trainable:,}')
    print(f'Frozen parameters:      {non_trainable:,}')
    print(f'Frozen layers:          {freeze_layers} of 12 transformer layers + embeddings')

    # Compile with binary cross-entropy and Adam
    # Note: class_weight in model.fit() handles imbalance, not the loss function directly
    bert.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics   = [tf.keras.metrics.AUC(name='auc', curve='PR')]
    )

    return bert


# Class weight for imbalance -- equivalent to scale_pos_weight
# class 0 (negative): weight 1.0, class 1 (positive): weight ~55
class_weight = {
    0: 1.0,
    1: float((ytr_bert == 0).sum() / (ytr_bert == 1).sum())
}
print(f'Class weights: {class_weight}')
print(f'Positive class weight: {class_weight[1]:.1f}x negative')

## 6. Custom MCC Early Stopping Callback

Keras's built-in early stopping monitors loss or AUC. We want to stop on MCC — our primary metric.

This custom callback:
- Computes MCC on the validation set after each epoch
- Tracks the best MCC seen so far
- Stops training if MCC does not improve for `patience` epochs
- Restores the best weights when stopping

In [ ]:
class MCCEarlyStopping(tf.keras.callbacks.Callback):
    """
    Custom Keras callback that monitors validation MCC.
    Stops training when MCC stops improving.
    Restores best weights on stop.
    """

    def __init__(self, validation_data, patience=2):
        super().__init__()
        self.validation_data = validation_data  # (dataset, true_labels)
        self.patience        = patience
        self.best_mcc        = -np.inf
        self.wait            = 0
        self.best_weights    = None

    def on_epoch_end(self, epoch, logs=None):
        val_dataset, val_labels = self.validation_data

        # Get predicted probabilities from validation set
        logits = self.model.predict(val_dataset, verbose=0)
        probs  = tf.sigmoid(logits).numpy().flatten()

        # Find MCC-maximizing threshold on validation predictions
        thresholds = np.linspace(0.01, 0.99, 100)
        mcc_vals   = [matthews_corrcoef(val_labels, (probs >= t).astype(int))
                      for t in thresholds]
        val_mcc    = max(mcc_vals)

        print(f'  Epoch {epoch+1} -- val MCC: {val_mcc:.4f}')

        if val_mcc > self.best_mcc:
            self.best_mcc    = val_mcc
            self.best_weights = self.model.get_weights()
            self.wait         = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)
                print(f'  Early stopping at epoch {epoch+1}. Best MCC: {self.best_mcc:.4f}')


print('MCCEarlyStopping callback defined.')

## 7. Stage 1 — ClinicalBERT OOF Fine-tuning

Run 5-fold stratified cross-validation on the training set.

**For each fold:**
1. Fine-tune ClinicalBERT on fold training notes
2. Predict the fold validation notes
3. Store these as OOF predictions

Each training patient is predicted by a model that never saw them during training. These OOF predictions become the `bert_note_prob` feature for XGBoost training.

**Max 10 epochs** with early stopping on val MCC (patience=2).

Note: This is the most time-intensive step. With GPU expect ~15-20 min total. On CPU expect several hours.

In [ ]:
MAX_EPOCHS  = 10
FREEZE_LAYERS = 10
LEARNING_RATE = 2e-5

skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probs    = np.zeros(len(ytr_bert))   # will fill fold by fold

print('Stage 1: ClinicalBERT OOF fine-tuning (5 folds)')
print(f'Max epochs: {MAX_EPOCHS} | Early stopping patience: 2 | Freeze layers: {FREEZE_LAYERS}')
print('=' * 60)

for fold_num, (fold_tr_idx, fold_val_idx) in enumerate(
        skf.split(train_ids, ytr_bert)):

    print(f'\nFold {fold_num+1}/5')
    print(f'  Train: {len(fold_tr_idx)} | Val: {len(fold_val_idx)} '
          f'| Val positives: {ytr_bert[fold_val_idx].sum()}')

    # Subset tokenized data for this fold
    f_tr_ids   = train_ids[fold_tr_idx]
    f_tr_masks = train_masks[fold_tr_idx]
    f_tr_labels= ytr_bert[fold_tr_idx]

    f_val_ids   = train_ids[fold_val_idx]
    f_val_masks = train_masks[fold_val_idx]
    f_val_labels= ytr_bert[fold_val_idx]

    # Build TF datasets
    train_ds = make_tf_dataset(f_tr_ids, f_tr_masks, f_tr_labels,
                               batch_size=BATCH_SIZE, shuffle=True)
    val_ds   = make_tf_dataset(f_val_ids, f_val_masks,
                               batch_size=BATCH_SIZE, shuffle=False)

    # Build fresh model for this fold
    tf.keras.backend.clear_session()
    model = build_bert_model(BERT_MODEL_NAME, FREEZE_LAYERS, LEARNING_RATE)

    # Class weight for this fold's training labels
    fold_neg = (f_tr_labels == 0).sum()
    fold_pos = (f_tr_labels == 1).sum()
    fold_class_weight = {0: 1.0, 1: float(fold_neg / fold_pos)}

    # MCC early stopping callback
    mcc_callback = MCCEarlyStopping(
        validation_data = (val_ds, f_val_labels),
        patience        = 2
    )

    # Fine-tune
    model.fit(
        train_ds,
        epochs       = MAX_EPOCHS,
        class_weight = fold_class_weight,
        callbacks    = [mcc_callback],
        verbose      = 0
    )

    # Predict fold validation patients (OOF)
    logits = model.predict(val_ds, verbose=0)
    fold_probs = tf.sigmoid(logits).numpy().flatten()
    oof_probs[fold_val_idx] = fold_probs

    fold_pr_auc = average_precision_score(f_val_labels, fold_probs)
    print(f'  Fold {fold_num+1} OOF PR-AUC: {fold_pr_auc:.4f}')

    # Free memory
    del model, train_ds, val_ds
    tf.keras.backend.clear_session()

print()
print(f'OOF complete. Overall OOF PR-AUC: {average_precision_score(ytr_bert, oof_probs):.4f}')
print(f'OOF MCC (at 0.5): {matthews_corrcoef(ytr_bert, (oof_probs>=0.5).astype(int)):.4f}')

## 8. Train Final ClinicalBERT on Full Training Set

After OOF, train one final BERT model on all training data. This model is used to predict the test set.

Early stopping uses a 15% internal validation split of the training data (not the test set).

In [ ]:
print('Training final ClinicalBERT on full training set...')

# Use 15% of training data as internal val for early stopping
# This is separate from the test set -- test set never touched
n_train_final = len(train_ids)
n_val_final   = int(n_train_final * 0.15)
val_final_idx = np.random.choice(n_train_final, n_val_final, replace=False)
tr_final_idx  = np.setdiff1d(np.arange(n_train_final), val_final_idx)

final_tr_ds = make_tf_dataset(
    train_ids[tr_final_idx], train_masks[tr_final_idx], ytr_bert[tr_final_idx],
    batch_size=BATCH_SIZE, shuffle=True
)
final_val_ds = make_tf_dataset(
    train_ids[val_final_idx], train_masks[val_final_idx],
    batch_size=BATCH_SIZE, shuffle=False
)
test_ds = make_tf_dataset(
    test_ids, test_masks,
    batch_size=BATCH_SIZE, shuffle=False
)

tf.keras.backend.clear_session()
final_bert = build_bert_model(BERT_MODEL_NAME, FREEZE_LAYERS, LEARNING_RATE)

mcc_callback_final = MCCEarlyStopping(
    validation_data = (final_val_ds, ytr_bert[val_final_idx]),
    patience        = 2
)

final_bert.fit(
    final_tr_ds,
    epochs       = MAX_EPOCHS,
    class_weight = class_weight,
    callbacks    = [mcc_callback_final],
    verbose      = 0
)

# Predict test set
test_logits    = final_bert.predict(test_ds, verbose=0)
test_bert_prob = tf.sigmoid(test_logits).numpy().flatten()

print(f'Final BERT -- Test PR-AUC (notes only): '
      f'{average_precision_score(yte_bert, test_bert_prob):.4f}')
print(f'This is BERT standalone performance on notes alone.')
print('Combining with tabular features in Stage 2 should improve this further.')

## 9. Add bert_note_prob as Feature 140

The OOF BERT probabilities become the `bert_note_prob` feature for XGBoost training. The test BERT probabilities become the same feature for the test set.

**Why one feature and not 768:**
XGBoost splits on individual dimensions one at a time. BERT embedding dimensions are dense and entangled -- no single dimension has independent meaning. Compressing to one probability preserves the semantic signal in a form XGBoost can use correctly via threshold splits.

In [ ]:
# Add bert_note_prob as column 140 to tabular feature matrices
Xtr_full = Xtr_tab.copy()
Xte_full = Xte_tab.copy()

Xtr_full['bert_note_prob'] = oof_probs       # OOF predictions -- no leakage
Xte_full['bert_note_prob'] = test_bert_prob  # final BERT predictions on test

feature_cols_full = fc + ['bert_note_prob']

print(f'Feature matrix shape -- train: {Xtr_full.shape}')
print(f'Feature matrix shape -- test:  {Xte_full.shape}')
print(f'Total features: {len(feature_cols_full)} (139 tabular + 1 BERT probability)')
print()
print('bert_note_prob stats (train):')
print(f'  Mean:   {oof_probs.mean():.4f}')
print(f'  Median: {np.median(oof_probs):.4f}')
print(f'  Positive class mean: {oof_probs[ytr_bert==1].mean():.4f}')
print(f'  Negative class mean: {oof_probs[ytr_bert==0].mean():.4f}')
print()
print('Good separation between positive/negative BERT probs = notes add signal.')

## 10. Stage 2 — XGBoost Pass 1 (140 features)

Train XGBoost on the full 140-feature matrix (139 tabular + bert_note_prob).

**Optuna:** 20 trials, 3-fold stratified CV on train set only. 8 hyperparameters including L1/L2 regularization (these were in the best params that gave 34/36 recall).

**After Pass 1:** SHAP analysis identifies which features the model actually uses. Features with mean |SHAP| >= 0.05 survive to Pass 2.

In [ ]:
def xgb_objective(trial, X_tr, y_tr):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
    }
    skf_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores  = []
    for ti, vi in skf_xgb.split(X_tr, y_tr):
        Xf = X_tr.iloc[ti]; Xv = X_tr.iloc[vi]
        yf = y_tr[ti];      yv = y_tr[vi]
        fold_spw = (yf==0).sum() / (yf==1).sum()
        m = xgb.XGBClassifier(**params, scale_pos_weight=fold_spw,
                               eval_metric='aucpr', random_state=SEED,
                               n_jobs=2, verbosity=0)
        m.fit(Xf, yf)
        scores.append(average_precision_score(yv, m.predict_proba(Xv)[:,1]))
    return np.mean(scores)


print('XGBoost Pass 1 -- Optuna (20 trials, 3-fold CV)...')
study1 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=SEED))
study1.optimize(lambda t: xgb_objective(t, Xtr_full, ytr),
                n_trials=20, show_progress_bar=False)
p1_params = study1.best_params
print(f'Pass 1 best CV PR-AUC: {study1.best_value:.4f}')
print(f'Best params: {p1_params}')

# Train on full training set
model_p1 = xgb.XGBClassifier(**p1_params, scale_pos_weight=spw,
                               eval_metric='aucpr', random_state=SEED,
                               n_jobs=2, verbosity=0)
model_p1.fit(Xtr_full, ytr)
print('Pass 1 model trained.')

## 11. SHAP Feature Selection

Compute SHAP values on Pass 1 model to identify which features contribute real signal.

**Key question:** Does `bert_note_prob` rank high in SHAP?
- If yes -- notes add signal beyond labs. ClinicalBERT pipeline is valuable.
- If near zero -- lab values already capture everything. Notes add no incremental signal.

Either outcome is informative and guides next steps.

In [ ]:
print('Computing SHAP values (sample 1500 rows)...')
np.random.seed(SEED)
sample_idx  = np.random.choice(len(Xtr_full), size=1500, replace=False)
Xtr_sample  = Xtr_full.iloc[sample_idx]

explainer   = shap.TreeExplainer(model_p1)
shap_values = explainer.shap_values(Xtr_sample)

mean_shap = pd.DataFrame({
    'feature':       feature_cols_full,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features by SHAP importance:')
print(mean_shap.head(15)[['feature','mean_abs_shap']].to_string(index=False))
print()

# Find bert_note_prob rank
bert_rank = mean_shap[mean_shap['feature']=='bert_note_prob'].index[0] + 1
bert_shap = mean_shap[mean_shap['feature']=='bert_note_prob']['mean_abs_shap'].values[0]
print(f'bert_note_prob rank: {bert_rank} of {len(feature_cols_full)}')
print(f'bert_note_prob SHAP: {bert_shap:.4f}')
print()
if bert_shap >= 0.05:
    print('bert_note_prob survived SHAP threshold (>=0.05).')
    print('Notes add meaningful signal beyond lab values.')
else:
    print('bert_note_prob below SHAP threshold (<0.05).')
    print('Notes add minimal incremental signal given the lab features.')
    print('This is a valid finding -- labs already capture the signal.')

# Select Pass 2 features
SHAP_THRESHOLD  = 0.05
pass2_features  = mean_shap[mean_shap['mean_abs_shap'] >= SHAP_THRESHOLD]['feature'].tolist()
print(f'\nFeatures selected for Pass 2: {len(pass2_features)}')
print(f'Features dropped: {len(feature_cols_full) - len(pass2_features)}')
print(f'EPV: {ytr.sum()}/{len(pass2_features)} = {ytr.sum()/len(pass2_features):.1f}')

## 12. XGBoost Pass 2 — SHAP-selected Features

Retrain XGBoost on the SHAP-selected feature set.

**OOF MCC threshold:**
Run 3-fold OOF on training set. Find threshold maximizing MCC on OOF predictions. Apply fixed threshold to test set.

**Recall floor alternative (commented out):**
If clinical priority is catching as many cases as possible, use the recall >= 0.80 approach instead.

In [ ]:
Xtr_p2 = Xtr_full[pass2_features]
Xte_p2 = Xte_full[pass2_features]

print(f'Pass 2 Optuna ({len(pass2_features)} features)...')
study2 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=SEED))
study2.optimize(lambda t: xgb_objective(t, Xtr_p2, ytr),
                n_trials=20, show_progress_bar=False)
p2_params = study2.best_params
print(f'Pass 2 best CV PR-AUC: {study2.best_value:.4f}')

# Train final Pass 2 model
model_p2 = xgb.XGBClassifier(**p2_params, scale_pos_weight=spw,
                               eval_metric='aucpr', random_state=SEED,
                               n_jobs=2, verbosity=0)
model_p2.fit(Xtr_p2, ytr)

# OOF predictions for MCC threshold selection
skf_oof = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_xgb = np.zeros(len(ytr))
for ti, vi in skf_oof.split(Xtr_p2, ytr):
    Xf = Xtr_p2.iloc[ti]; Xv = Xtr_p2.iloc[vi]; yf = ytr[ti]
    fold_spw = (yf==0).sum()/(yf==1).sum()
    m = xgb.XGBClassifier(**p2_params, scale_pos_weight=fold_spw,
                           eval_metric='aucpr', random_state=SEED,
                           n_jobs=2, verbosity=0)
    m.fit(Xf, yf)
    oof_xgb[vi] = m.predict_proba(Xv)[:,1]

# OOF MCC threshold -- primary approach
thresholds = np.linspace(0.01, 0.99, 200)
mcc_scores = [matthews_corrcoef(ytr, (oof_xgb >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(mcc_scores)]
print(f'OOF MCC: {max(mcc_scores):.4f}  Threshold: {best_threshold:.4f}')

# ── Alternative: recall floor approach (uncomment to use instead) ────────────
# Use when catching as many cases as possible is the clinical priority
# (missing a case more costly than a false alarm -- gives recall ~0.94 vs ~0.64)
#
# prec_oof, rec_oof, thresh_oof = precision_recall_curve(ytr, oof_xgb)
# valid          = [(p,r,t) for p,r,t in zip(prec_oof,rec_oof,thresh_oof) if r>=0.80]
# best_threshold = max(valid, key=lambda x: x[0])[2] if valid else 0.5

## 13. Final Evaluation on Test Set

Test set is used **exactly once** here. All hyperparameter decisions, feature selection, and threshold selection were made using training data only.

In [ ]:
# Final test evaluation -- test set used exactly once
test_probs = model_p2.predict_proba(Xte_p2)[:,1]
test_preds = (test_probs >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(yte, test_preds).ravel()
mcc       = matthews_corrcoef(yte, test_preds)
prauc     = average_precision_score(yte, test_probs)
roc       = roc_auc_score(yte, test_probs)
recall    = tp / (tp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1        = f1_score(yte, test_preds, zero_division=0)

print('=' * 55)
print('  FINAL RESULTS -- ClinicalBERT + XGBoost Pass 2')
print('=' * 55)
print(f'  Features:  {len(pass2_features)} (SHAP-selected from 140)')
print(f'  MCC:       {mcc:.4f}  <- primary metric')
print(f'  PR-AUC:    {prauc:.4f}')
print(f'  ROC-AUC:   {roc:.4f}')
print(f'  Recall:    {recall:.4f}  ({tp} of {yte.sum()} caught)')
print(f'  Precision: {precision:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  Threshold: {best_threshold:.4f}  (OOF MCC-optimized)')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print()
print(f'bert_note_prob SHAP rank: {bert_rank} of {len(feature_cols_full)}')
print(f'bert_note_prob SHAP value: {bert_shap:.4f}')

## 14. Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confusion matrix
cm_arr = np.array([[tn, fp], [fn, tp]])
axes[0].imshow(cm_arr, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(['Pred Neg','Pred Pos'])
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(['Actual Neg','Actual Pos'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm_arr[i,j], ha='center', va='center',
                     fontsize=18, fontweight='bold',
                     color='white' if cm_arr[i,j]>500 else 'black')
axes[0].set_title(f'Confusion Matrix\nMCC={mcc:.3f}  Threshold={best_threshold:.3f}')

# PR curve
pc, rc, _ = precision_recall_curve(yte, test_probs)
axes[1].plot(rc, pc, color='#185FA5', linewidth=2)
axes[1].axhline(yte.mean(), color='gray', linestyle=':', linewidth=1, label='Baseline')
axes[1].scatter([recall], [precision], color='#D85A30', s=100, zorder=5,
                label=f'Threshold={best_threshold:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR Curve  PR-AUC={prauc:.3f}')
axes[1].legend(fontsize=8)

# ROC curve
fpr, tpr, _ = roc_curve(yte, test_probs)
axes[2].plot(fpr, tpr, color='#185FA5', linewidth=2)
axes[2].plot([0,1],[0,1], color='gray', linestyle=':')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title(f'ROC Curve  AUC={roc:.3f}')

plt.suptitle(
    f'ClinicalBERT + XGBoost Pass 2 ({len(pass2_features)} features) -- Final Results',
    y=1.02)
plt.tight_layout()
plt.savefig('bert_xgb_final_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved: bert_xgb_final_results.png')

## 15. SHAP Summary Plot

Shows which features drove the final XGBoost Pass 2 predictions. Look for `bert_note_prob` position -- this tells you how much the notes contributed vs the lab values.

In [ ]:
print('Computing SHAP for Pass 2 model...')
np.random.seed(SEED)
sample_idx2  = np.random.choice(len(Xtr_p2), size=1500, replace=False)
Xtr_p2_samp  = Xtr_p2.iloc[sample_idx2]
explainer_p2 = shap.TreeExplainer(model_p2)
shap_vals_p2 = explainer_p2.shap_values(Xtr_p2_samp)

mean_shap_p2 = pd.DataFrame({
    'feature':       pass2_features,
    'mean_abs_shap': np.abs(shap_vals_p2).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features -- Pass 2 model:')
print(mean_shap_p2.head(15)[['feature','mean_abs_shap']].to_string(index=False))

# SHAP bar chart -- top 15
top15 = mean_shap_p2.head(15)
colors = []
for feat in top15['feature']:
    if feat == 'bert_note_prob':
        colors.append('#E8A838')  # gold -- highlight BERT feature
    elif any(feat.startswith(l) for l in ['wbcNum','rbcNum','plateletNum']):
        colors.append('#185FA5')  # blue -- key labs
    elif feat.startswith('drug_') or feat in ['total_admin_events',
                                               'total_distinct_drugs','meds_missing']:
        colors.append('#D85A30')  # orange -- medications
    else:
        colors.append('#888780')  # grey -- other

fig2, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top15)), top15['mean_abs_shap'], color=colors, height=0.7)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature importance -- ClinicalBERT + XGBoost Pass 2\n'
             'Gold = BERT note probability  |  Blue = key labs  |  Grey = other')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#E8A838', label='bert_note_prob (ClinicalBERT)'),
    Patch(facecolor='#185FA5', label='Key labs (WBC/RBC/Platelet)'),
    Patch(facecolor='#D85A30', label='Medications'),
    Patch(facecolor='#888780', label='Other labs / Demographics'),
], fontsize=8)

plt.tight_layout()
plt.savefig('bert_xgb_shap.png', dpi=130, bbox_inches='tight')
plt.show()
print('SHAP plot saved: bert_xgb_shap.png')

## 16. Pipeline Summary

In [ ]:
print('=' * 60)
print('PIPELINE SUMMARY -- ClinicalBERT + XGBoost')
print('=' * 60)
print()
print('Architecture:')
print('  Stage 1: Fine-tuned ClinicalBERT (Bio_ClinicalBERT)')
print(f'    Frozen layers:    {FREEZE_LAYERS} of 12')
print(f'    Max epochs:       {MAX_EPOCHS} with MCC early stopping')
print(f'    OOF folds:        5 (same splits as XGBoost)')
print(f'    BERT standalone PR-AUC (test): '
      f'{average_precision_score(yte_bert, test_bert_prob):.4f}')
print()
print('  Stage 2: XGBoost Pass 2')
print(f'    Features:         {len(pass2_features)} (SHAP-selected from 140)')
print(f'    bert_note_prob rank: {bert_rank} of {len(feature_cols_full)}')
print(f'    bert_note_prob SHAP: {bert_shap:.4f}')
print()
print('Final results (test set):')
print(f'  MCC:       {mcc:.4f}')
print(f'  PR-AUC:    {prauc:.4f}')
print(f'  ROC-AUC:   {roc:.4f}')
print(f'  Recall:    {recall:.4f}  ({tp}/{yte.sum()} caught)')
print(f'  Precision: {precision:.4f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print()
print('Leakage controls:')
print('  BERT OOF: each train patient predicted by model that never saw them')
print('  Same 5-fold splits used for BERT and XGBoost (SEED=42)')
print('  XGBoost threshold from OOF MCC -- not test set')
print('  Test set used exactly once for final reporting')